# Setup

In [ ]:
import shutil, os
if os.path.exists('e_network_inequality'):
    shutil.rmtree('e_network_inequality')

# !git clone https://github.com/IgnacioOQ/e_network_inequality
!git clone -b main https://github.com/IgnacioOQ/e_network_inequality

In [ ]:
!pip install dill

In [ ]:
%cd e_network_inequality

/content/e_network_inequality/e_network_inequality


In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

from utils.imports import *
from model.agents import BetaAgent, BayesAgent
from model.model import Model
from utils.network_utils import *
from networks.network_generation import *
from networks.variation_methods import *
from model.simulation_functions import *
from model.vectorized_simulation_functions import *
from functools import partial
import hashlib
import gc
from multiprocessing import get_context

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

dumping_path = '/content/drive/My Drive/Colab Projects/Data Driven ABMs/Data Sets/stop_condition_study/'
print("Current Directory:", dumping_path)

Mounted at /content/drive
Current Directory: /content/drive/My Drive/Colab Projects/Data Driven ABMs/Data Sets/


In [ ]:
def generate_parameters_here(_,G,method='randomization'):
    process_seed = int.from_bytes(os.urandom(4), byteorder='little')
    rd.seed(process_seed)
    # Randomly sample parameters for this group
    uncertainty = rd.uniform(.000001, .001)
    n_experiments = rd.randint(1000, 10000)
    # now we pick a random number
    # Capped at 1/3 to prevent "Sample larger than population" errors in equalize
    proportion_edges = rd.rand() * (1/3)
    # Do randomization
    num_edges = G.number_of_edges()
    if method == 'randomization':
      num_edges_to_randomize = int(num_edges * proportion_edges)
      modified_network = randomize_network(G, n_edges=num_edges_to_randomize)
    if method == 'equalize':
      num_edges_to_randomize = int(num_edges * proportion_edges)
      modified_network = equalize(G, num_edges_to_randomize)
    if method == 'densify':
      num_edges_to_add = int(num_edges * proportion_edges)
      modified_network = densify_fancy_speed_up(G,num_edges_to_add,target_degree_dist='original',keep_density_fixed=False)
    if method == 'densify_fixed':
      num_edges_to_add = int(num_edges * proportion_edges)
      modified_network = densify_fancy_speed_up(G,num_edges_to_add,target_degree_dist='uniform',keep_density_fixed=True)
    if method =='cluster':
      num_edges_to_add = int(num_edges * proportion_edges)
      modified_network = cluster_network(G,num_edges_to_add)
    if method =='decluster':
      num_edges_to_randomize = int(num_edges * proportion_edges)
      modified_network = decluster_network(G,num_edges_to_randomize)

    result = generate_parameters_aggregate(modified_network, uncertainty=uncertainty, n_experiments=n_experiments,
                                           p_rewiring=proportion_edges)
    result['uncertainty'] = uncertainty
    result['n_experiments'] = n_experiments
    result['proportion_edges'] = proportion_edges
    result['parameter_random_seed']= process_seed
    return result

# Study 0: Load Networks

In [ ]:
# ── Load Networks + Parallelization Setup ────────────────────────────────────
from model.vectorized_model import VectorizedModel
from model.vectorized_simulation_functions import run_vectorized_simulation_with_params
from functools import partial
from multiprocessing import Pool, cpu_count
import itertools
import pickle
import networkx as nx

num_cores = cpu_count()
print(f"Available CPU cores: {num_cores}")

with open('./networks/citation_data/pud_network.pkl', 'rb') as f:
    G_pud = pickle.load(f)
mapping = {node: idx for idx, node in enumerate(G_pud.nodes())}
G_pud_indexed = nx.relabel_nodes(G_pud, mapping)
print(f"PUD network: {G_pud_indexed.number_of_nodes()} nodes, {G_pud_indexed.number_of_edges()} edges")

with open('./networks/citation_data/tobacco_network.pkl', 'rb') as f:
    G_tobacco = pickle.load(f)
mapping = {node: idx for idx, node in enumerate(G_tobacco.nodes())}
G_tobacco_indexed = nx.relabel_nodes(G_tobacco, mapping)
print(f"Tobacco network: {G_tobacco_indexed.number_of_nodes()} nodes, {G_tobacco_indexed.number_of_edges()} edges")

with open('./networks/citation_data/ego_network.pkl', 'rb') as f:
    G_ego = pickle.load(f)
mapping = {node: idx for idx, node in enumerate(G_ego.nodes())}
G_ego_indexed = nx.relabel_nodes(G_ego, mapping)
print(f"Ego network: {G_ego_indexed.number_of_nodes()} nodes, {G_ego_indexed.number_of_edges()} edges")

# Study 1: Parameter Search (Parallel Simulations)

In [ ]:
# ── Parameter Search — Config ─────────────────────────────────────────────────
# Grid: tolerance x uncertainty x n_experiments
# All combinations are flattened into a single param_dicts list, then
# dispatched in one Pool call per (n_experiments, tolerance) group for
# memory efficiency. Results are collected and saved per network.

PS_TOLERANCES    = [1e-3, 1e-4, 1e-5, 1e-6]
PS_UNCERTAINTIES = [0.0001, 0.001, 0.005, 0.01]
PS_N_EXPERIMENTS = [100, 500, 1000]
PS_N_RUNS        = 500
PS_MAX_STEPS     = 100_000

In [ ]:
def run_parameter_search_parallel(network, network_label, output_prefix):
    import time

    combos = list(itertools.product(PS_N_EXPERIMENTS, PS_UNCERTAINTIES, PS_TOLERANCES))
    n_combos = len(combos)
    total_sims = n_combos * PS_N_RUNS
    print(f"  [{network_label}] {n_combos} combos × {PS_N_RUNS} runs = {total_sims:,} simulations")

    rows = []
    t_total_start = time.time()

    with tqdm(combos, desc=f"[{network_label}] combos", unit="combo") as pbar:
        for n_exp, uncertainty, tolerance in pbar:
            param_dicts = [
                {"network":      network,
                 "n_experiments": n_exp,
                 "uncertainty":  uncertainty,
                 "seed":         seed}
                for seed in range(PS_N_RUNS)
            ]

            wrapper = partial(
                run_vectorized_simulation_with_params,
                tolerance=tolerance,
                tolerance_stopping=True,
                tstep_stopping=False,
                number_of_steps=PS_MAX_STEPS,
                show_bar=False,
                agent_type="beta",
            )

            t0 = time.time()
            with Pool(num_cores) as pool:
                results = list(tqdm(
                    pool.imap_unordered(wrapper, param_dicts),
                    total=PS_N_RUNS,
                    desc=f"  n_exp={n_exp} unc={uncertainty:.4f} tol={tolerance:.0e}",
                    leave=False,
                ))
            group_elapsed = time.time() - t0

            for r in results:
                rows.append({
                    "tolerance":     tolerance,
                    "uncertainty":   uncertainty,
                    "n_experiments": n_exp,
                    "steps":         r["convergence_step"],
                    "truth_share":   r["share_of_correct_agents_at_convergence"],
                    "hit_cap":       int(r["convergence_step"] == PS_MAX_STEPS),
                })

            pbar.set_postfix(
                last=f"{group_elapsed:.1f}s",
                sims=f"{len(rows):,}/{total_sims:,}",
                tol=f"{tolerance:.0e}",
            )

    total_elapsed = time.time() - t_total_start
    print(f"  [{network_label}] Done — {total_sims:,} sims in "
          f"{total_elapsed/60:.1f} min ({total_elapsed:.0f}s)")

    df_ps = pd.DataFrame(rows)
    raw_path = dumping_path + f"{output_prefix}_parameter_search.csv"
    df_ps.to_csv(raw_path, index=False)
    print(f"  Raw data saved: {raw_path} ({len(df_ps)} rows)")

    # Summary
    summary = (
        df_ps.groupby(["n_experiments", "uncertainty", "tolerance"])
        .agg(mean_steps=("steps", "mean"), std_steps=("steps", "std"),
             mean_truth=("truth_share", "mean"), std_truth=("truth_share", "std"),
             hit_cap_frac=("hit_cap", "mean"))
        .reset_index()
    )
    sum_path = dumping_path + f"{output_prefix}_parameter_search_summary.csv"
    summary.to_csv(sum_path, index=False)
    print(f"  Summary saved: {sum_path}")

    # Heatmaps (one per n_experiments value)
    tol_labels = [f"{t:.0e}" for t in PS_TOLERANCES]
    unc_labels = [str(u) for u in PS_UNCERTAINTIES]
    for n_exp_val in PS_N_EXPERIMENTS:
        df_sub = df_ps[df_ps.n_experiments == n_exp_val]
        mean_steps = np.zeros((len(PS_UNCERTAINTIES), len(PS_TOLERANCES)))
        std_steps  = np.zeros_like(mean_steps)
        mean_truth = np.zeros_like(mean_steps)
        std_truth  = np.zeros_like(mean_steps)
        for i, unc in enumerate(PS_UNCERTAINTIES):
            for j, tol in enumerate(PS_TOLERANCES):
                sub = df_sub[(df_sub.uncertainty == unc) & (df_sub.tolerance == tol)]
                mean_steps[i, j] = sub["steps"].mean()
                std_steps[i, j]  = sub["steps"].std()
                mean_truth[i, j] = sub["truth_share"].mean()
                std_truth[i, j]  = sub["truth_share"].std()
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle(f"Parameter Search - {network_label}, "
                     f"n_experiments={n_exp_val} ({PS_N_RUNS} runs)", fontsize=13)
        panels = [
            (axes[0, 0], mean_steps, "Mean steps to convergence", "YlOrRd"),
            (axes[0, 1], std_steps,  "Std dev of steps",          "YlOrRd"),
            (axes[1, 0], mean_truth, "Mean truth share",          "RdYlGn"),
            (axes[1, 1], std_truth,  "Std dev of truth share",    "RdYlGn_r"),
        ]
        for ax, data, title, cmap in panels:
            im = ax.imshow(data, aspect='auto', cmap=cmap)
            ax.set_xticks(range(len(PS_TOLERANCES)))
            ax.set_xticklabels(tol_labels, fontsize=9)
            ax.set_yticks(range(len(PS_UNCERTAINTIES)))
            ax.set_yticklabels(unc_labels, fontsize=9)
            ax.set_xlabel("Tolerance", fontsize=10)
            ax.set_ylabel("Uncertainty", fontsize=10)
            ax.set_title(title, fontsize=11)
            plt.colorbar(im, ax=ax)
            for i in range(len(PS_UNCERTAINTIES)):
                for j in range(len(PS_TOLERANCES)):
                    val = data[i, j]
                    fmt = f"{val:.0f}" if "steps" in title.lower() else f"{val:.3f}"
                    rng = data.max() - data.min() + 1e-9
                    color = 'black' if 0.2 < (val - data.min()) / rng < 0.8 else 'white'
                    ax.text(j, i, fmt, ha='center', va='center',
                            fontsize=7, color=color)
        plt.tight_layout()
        hm_path = (dumping_path
                   + f"{output_prefix}_parameter_search_heatmap_nexp{n_exp_val}.png")
        plt.savefig(hm_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"  Saved: {hm_path}")

    # Line plots: mean steps vs tolerance, faceted by n_experiments
    fig, axes = plt.subplots(1, len(PS_N_EXPERIMENTS),
                             figsize=(5 * len(PS_N_EXPERIMENTS), 5), sharey=True)
    colors = plt.cm.viridis(np.linspace(0, 0.9, len(PS_UNCERTAINTIES)))
    for ax, n_exp_val in zip(axes, PS_N_EXPERIMENTS):
        df_sub = df_ps[df_ps.n_experiments == n_exp_val]
        for unc, color in zip(PS_UNCERTAINTIES, colors):
            means = [
                df_sub[(df_sub.uncertainty == unc)
                       & (df_sub.tolerance == tol)]["steps"].mean()
                for tol in PS_TOLERANCES
            ]
            ax.plot(tol_labels, means, 'o-', color=color,
                    label=f"unc={unc}", linewidth=1.8, markersize=5)
        ax.set_yscale('log')
        ax.set_title(f"n_experiments={n_exp_val}", fontsize=11)
        ax.set_xlabel("Tolerance", fontsize=10)
        ax.grid(True, alpha=0.3)
        if ax is axes[0]:
            ax.set_ylabel("Mean steps (log)", fontsize=10)
    axes[-1].legend(fontsize=9, bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.suptitle(f"Convergence Speed: Steps vs. Tolerance - {network_label}",
                 fontsize=12)
    plt.tight_layout()
    lines_path = dumping_path + f"{output_prefix}_parameter_search_lines.png"
    plt.savefig(lines_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close('all')
    print(f"  Saved: {lines_path}")
    # Line plots: mean truth share vs tolerance, faceted by n_experiments
    fig, axes = plt.subplots(1, len(PS_N_EXPERIMENTS),
                             figsize=(5 * len(PS_N_EXPERIMENTS), 5), sharey=True)
    colors = plt.cm.plasma(np.linspace(0, 0.85, len(PS_UNCERTAINTIES)))
    for ax, n_exp_val in zip(axes, PS_N_EXPERIMENTS):
        df_sub = df_ps[df_ps.n_experiments == n_exp_val]
        for unc, color in zip(PS_UNCERTAINTIES, colors):
            truth_means = [
                df_sub[(df_sub.uncertainty == unc)
                       & (df_sub.tolerance == tol)]["truth_share"].mean()
                for tol in PS_TOLERANCES
            ]
            ax.plot(tol_labels, truth_means, 'o-', color=color,
                    label=f"unc={unc}", linewidth=1.8, markersize=5)
        ax.set_ylim(0, 1)
        ax.set_title(f"n_experiments={n_exp_val}", fontsize=11)
        ax.set_xlabel("Tolerance", fontsize=10)
        ax.grid(True, alpha=0.3)
        if ax is axes[0]:
            ax.set_ylabel("Mean truth share", fontsize=10)
    axes[-1].legend(fontsize=9, bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.suptitle(f"Truth Share vs. Tolerance - {network_label}",
                 fontsize=12)
    plt.tight_layout()
    truth_lines_path = dumping_path + f"{output_prefix}_parameter_search_truth_lines.png"
    plt.savefig(truth_lines_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close('all')
    print(f"  Saved: {truth_lines_path}")

## PUD Network

In [ ]:
print("=== Parameter Search: PUD Network ===")
run_parameter_search_parallel(G_pud_indexed, "PUD", output_prefix="pud")

## Tobacco Network

In [ ]:
print("=== Parameter Search: Tobacco Network ===")
run_parameter_search_parallel(G_tobacco_indexed, "Tobacco", output_prefix="tobacco")

## Ego Network

In [ ]:
print("=== Parameter Search: Ego Network ===")
run_parameter_search_parallel(G_ego_indexed, "Ego", output_prefix="ego")

# Study 2: Parameter Search — Data Analysis

## Overview

This section loads the CSV files produced by Study 1 and applies linear
regression to quantify how much of the variance in two outcomes —
**steps to convergence** and **mean truth share** — is attributable to
each of the three input parameters.

### Hypotheses

- **H1 (Steps):** The number of steps to convergence is driven mainly by
  `tolerance`. `uncertainty` and `n_experiments` have little effect.
- **H2 (Truth share):** Mean truth share at convergence is driven mainly
  by `uncertainty`. `tolerance` and `n_experiments` have little effect.

### Log transforms

All three predictors span at least one order of magnitude in the grid:

| Predictor | Raw range | log₁₀ range |
|-----------|-----------|-------------|
| `tolerance` | 1e-3 → 1e-6 | −3 → −6 |
| `uncertainty` | 1e-4 → 1e-2 | −4 → −2 |
| `n_experiments` | 100 → 500 | 2.0 → 2.7 |

Log-transforming makes coefficients interpretable and avoids numerical
instability from near-zero raw tolerance values.

### Sign convention

- `log_tolerance`: more negative = stricter tolerance. A **negative**
  coefficient on `steps` means stricter tolerance → more steps.
- `log_uncertainty`: more negative = smaller bandit gap. A **positive**
  coefficient on `truth_share` means larger gap → higher truth share.

### Multicollinearity note

Because Study 2 uses a fully factorial (orthogonal) grid, we expect
VIF ≈ 1.0 for all predictors. The multicollinearity section below
verifies this — it is a sanity check, not a diagnostic concern.

In [ ]:
from utils.data_analysis_utils import (
    compute_correlations, compute_vif,
    run_ols, regression_diagnostics, plot_f2_comparison,
)


In [ ]:
def analyze_parameter_search(network_label, output_prefix):
    # Load raw CSV produced by Study 2
    df = pd.read_csv(dumping_path + f"{output_prefix}_parameter_search.csv")

    # Log-transform all three predictors (they span orders of magnitude)
    df['log_tolerance']     = np.log10(df['tolerance'])      # range: -3 to ≈-6.3
    df['log_uncertainty']   = np.log10(df['uncertainty'])    # range: -4 to -2
    df['log_n_experiments'] = np.log10(df['n_experiments'])  # range: 2 to 3

    predictors = ['log_tolerance', 'log_uncertainty', 'log_n_experiments']

    print(f"\n{'='*60}")
    print(f"  Parameter Search Analysis: {network_label}  (n={len(df)})")
    print(f"{'='*60}\n")

    # ── 1. Multicollinearity study ─────────────────────────────────────────
    print("── Multicollinearity: Pearson correlations ──")
    compute_correlations(df, predictors)

    print("\n── Multicollinearity: Variance Inflation Factors ──")
    display(compute_vif(df, predictors))

    # ── 2. Regression: steps ~ predictors  (H1: tolerance dominates) ───────
    print("\n── Regression: Steps to Convergence ──")
    model_steps = run_ols(
        df, predictors, dependent='steps',
        label='Steps to Convergence', network_label=network_label,
    )
    regression_diagnostics(
        model_steps, title=f"Diagnostics: Steps to Convergence — {network_label}"
    )

    # ── 3. Regression: truth_share ~ predictors  (H2: uncertainty dominates) 
    print("\n── Regression: Mean Truth Share ──")
    model_truth = run_ols(
        df, predictors, dependent='truth_share',
        label='Mean Truth Share', network_label=network_label,
    )
    regression_diagnostics(
        model_truth, title=f"Diagnostics: Mean Truth Share — {network_label}"
    )

    # ── 4. Side-by-side f² comparison ─────────────────────────────────────
    plot_f2_comparison(
        models={
            'Steps to Convergence': model_steps,
            'Mean Truth Share':     model_truth,
        },
        predictors=predictors,
        title=f"Cohen's f² — Effect Size Comparison: {network_label}",
    )


## PUD Network

In [ ]:
print("=== Parameter Search Data Analysis: PUD Network ===")
analyze_parameter_search("PUD", "pud")

## Tobacco Network

In [ ]:
print("=== Parameter Search Data Analysis: Tobacco Network ===")
analyze_parameter_search("Tobacco", "tobacco")

## Ego Network

In [ ]:
print("=== Parameter Search Data Analysis: Ego Network ===")
analyze_parameter_search("Ego", "ego")

# Disconnect from Runtime

In [ ]:
from datetime import datetime
import pytz
from IPython.display import Javascript

# Get current time in New York
nyc_time = datetime.now(pytz.timezone('America/New_York'))
formatted_time = nyc_time.strftime('%Y-%m-%d %H:%M:%S %Z')

# Print and log
print(f"✅ Disconnected from runtime at: {formatted_time}")

# Disconnect Colab runtime
display(Javascript('google.colab.kernel.disconnect()'))

✅ Disconnected from runtime at: 2025-10-02 11:18:40 EDT


<IPython.core.display.Javascript object>